# Semantic Segmentation on CamVid: DeepLabV3+ vs. SegFormer-B2
### Comprehensive Machine Learning Project & NeurIPS 2023 JML Benchmark Study

1. **Traditional Baseline**: Random Forest on handcrafted spatial, color, and gradient features.
2. **Deep Models**: DeepLabV3+ (ResNet-101 backbone) vs. SegFormer-B2 (Mix Transformer mit-b2 backbone).
3. **Pretraining Ablation**: ImageNet vs. Cityscapes domain transfer on CE loss.
4. **5 Loss Functions Ablation**: Cross-Entropy, Lovász-Softmax, Dice, Focal, and JML (Jaccard Metric Loss).
5. **Goal-Driven Agent Search**: Bayesian optimization of composite loss weights $\lambda_{\text{CE}} \mathcal{L}_{\text{CE}} + \lambda_{\text{JML}} \mathcal{L}_{\text{JML}}$ on validation set.
6. **6 Systematic Improvements (3 Seeds)**: Native 720×960 Sliding Window Inference, Multi-Scale TTA Pyramid, Class Prior Calibration, Class-Aware Copy-Paste, and Cosine Annealing Extended Convergence.
7. **Final Test Evaluation**: Full metrics (mIoU, Dice, Precision, Recall, Pixel Acc, Boundary IoU, Confusion Matrices) with Mean ± Std across 3 seeds.

## 1. Environment Setup & Dependency Installation
Install all required high-performance computer vision and deep learning packages.

In [5]:
# Install dependencies in Google Colab
!pip install -q segmentation-models-pytorch transformers timm scikit-learn optuna albumentations opencv-python matplotlib seaborn pandas tabulate

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    torch.backends.cudnn.benchmark = True

PyTorch Version: 2.15.0+cpu
CUDA Available: False


In [ ]:
%pip install --upgrade jupyter ipywidgets tqdm

## 2. Dataset Download, Verification & Class Imbalance Analysis
Download official CamVid 11-class dataset (367 train, 101 val, 233 test) and quantify class imbalance.

In [6]:
import os
from pathlib import Path
from data.dataset import CamVidDataset, CAMVID_CLASSES, compute_dataset_statistics
from utils.visualization import plot_class_distribution
import pandas as pd

# Verify local dataset path
data_root = Path('./data/CamVid')
train_imgs = list(data_root.glob('**/train/**/*.png')) + list(data_root.glob('**/train/**/*.jpg'))

if len(train_imgs) >= 350:
    print(f'CamVid dataset already exists and verified at {data_root.resolve()} ({len(train_imgs)//2} images)! Skipping download.')
else:
    print('Dataset missing or incomplete, initiating download...')
    from data.download_camvid import setup_camvid_dataset
    setup_camvid_dataset(str(data_root))

# Compute class distribution to quantify severe class imbalance
train_set = CamVidDataset(str(data_root), split='train')
stats = compute_dataset_statistics(train_set)
plot_class_distribution(stats, save_path='./outputs/plots/00_class_pixel_imbalance.png')

df_stats = pd.DataFrame({
    'Class Name': stats['classes'],
    'Pixel Count': stats['pixel_counts'],
    'Frequency (%)': [f'{f*100:.3f}%' for f in stats['frequencies']],
    'Balanced Weight': [round(w, 3) for w in stats['class_weights']]
})
print(df_stats.to_string(index=False))


CamVid dataset already exists and verified at D:\finalprojv3\data\CamVid (367 images)! Skipping download.
Loaded CamVid [train] split: 367 images, 367 masks.
Computing class statistics on train set (367 images)...
Class distribution plot saved to D:\finalprojv3\outputs\plots\00_class_pixel_imbalance.png
Class Name  Pixel Count Frequency (%)  Balanced Weight
       Sky     10682767       17.538%            0.266
  Building     14750079       24.215%            0.193
      Pole       623349        1.023%            4.564
      Road     20076880       32.960%            0.142
  Pavement      2845085        4.671%            1.000
      Tree      6166762       10.124%            0.461
SignSymbol       743859        1.221%            3.825
     Fence       714595        1.173%            3.981
       Car      3719877        6.107%            0.765
Pedestrian       405385        0.666%            7.018
 Bicyclist       184967        0.304%           15.382


## 3. Traditional ML Baseline: Random Forest Classifier
Extract handcrafted features (RGB, HSV, normalized spatial coordinates, Sobel gradients, Gaussian texture blur) and evaluate on test set.

In [ ]:
# Run Random Forest baseline with unbuffered streaming output
!python -u experiments/run_baseline_rf.py --data_root ./data/CamVid --log_dir ./outputs/new_logs


In [56]:
import json, pandas as pd
from pathlib import Path

# Load RF summary
d = json.load(open(next(Path('./outputs').glob('**/*baseline_rf_summary.json'))))

# 1. Overall Metrics
display(pd.DataFrame([{
    'Model': 'Random Forest (50 Trees)',
    'mIoU': f"{d['mIoU']*100:.2f}%",
    'Pixel Acc': f"{d['pixel_accuracy']*100:.2f}%",
    'Dice/F1': f"{d['mean_dice_f1']*100:.2f}%",
    'Precision': f"{d['mean_precision']*100:.2f}%",
    'Recall': f"{d['mean_recall']*100:.2f}%",
    'Boundary IoU': f"{d['boundary_iou']:.4f}"
}]))

# 2. Per-Class IoU Breakdown
classes = ['Sky', 'Building', 'Pole', 'Road', 'Pavement', 'Tree', 'SignSymbol', 'Fence', 'Car', 'Pedestrian', 'Bicyclist']
display(pd.DataFrame([{'Class': classes[int(k)], 'IoU': f"{v*100:.2f}%"} for k, v in d['per_class_iou'].items()]))

,Model,mIoU,Pixel Acc,Dice/F1,Precision,Recall,Boundary IoU
0,Random Forest (50 Trees),30.85%,64.28%,41.05%,40.00%,48.41%,0.1142


,Class,IoU
0,Sky,88.72%
1,Building,30.67%
2,Pole,5.65%
3,Road,75.02%
4,Pavement,43.50%
5,Tree,31.57%
6,SignSymbol,9.83%
7,Fence,4.96%
8,Car,35.45%
9,Pedestrian,9.74%


## 4. Model Architectural Specifications, FLOPs & Inference Latency
Profile total parameters, trainable parameters, GFLOPs at 360×480, and latency in milliseconds.

In [9]:
# Auto-reload updated model definitions into the running kernel
import importlib
import pandas as pd
import models.deeplabv3plus
import models.model_factory
importlib.reload(models.deeplabv3plus)
importlib.reload(models.model_factory)
from models.model_factory import get_model_specs

specs_deeplab = get_model_specs('deeplabv3plus', pretraining='cityscapes')
specs_segformer = get_model_specs('segformer_b2', pretraining='cityscapes')

df_specs = pd.DataFrame([specs_deeplab, specs_segformer])
print('=== ARCHITECTURAL AND COMPUTATIONAL PROFILE ===')
display(df_specs[['model_name', 'total_parameters', 'trainable_parameters', 'gflops', 'avg_latency_ms', 'input_resolution']])


Loading Cityscapes domain-pretrained transfer weights for DeepLabV3+ ResNet-101...
Successfully transferred 624 layers from driving domain pretraining.
Initializing SegFormer-B2 from Cityscapes pretrained checkpoint: nvidia/segformer-b2-finetuned-cityscapes-1024-1024


[transformers] You passed `num_labels=11` which is incompatible to the `id2label` map of length `19`.
Loading weights: 100%|██████████| 380/380 [00:00<00:00, 47081.07it/s]
[transformers] SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b2-finetuned-cityscapes-1024-1024
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([19]) vs model:torch.Size([11])                      
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([19, 768, 1, 1]) vs model:torch.Size([11, 768, 1, 1])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


=== ARCHITECTURAL AND COMPUTATIONAL PROFILE ===


,model_name,total_parameters,trainable_parameters,gflops,avg_latency_ms,input_resolution
0,deeplabv3plus,45672283,45672283,60212.48,168.73,360x480
1,segformer_b2,27355083,27355083,36063.83,496.08,360x480


## 5. Pretraining Ablation: ImageNet vs. Cityscapes Domain Transfer
Evaluate transfer learning effectiveness by training both models (DeepLabV3+ and SegFormer-B2) under two distinct initialization regimes under standard Cross-Entropy loss:
1. **ImageNet-1K Initialization**: General-domain visual feature representation.
2. **Cityscapes Initialization**: In-domain urban driving scene representation.

> Each pretraining setting below is fully decoupled. You can run them independently in any order; results are safely appended and merged in `./outputs/new_logs/pretraining_ablation_summary.csv`.

### 5.1 Run Pretraining Regime #1: ImageNet Initialization (General-Domain)

In [ ]:
# 1.1 Run ImageNet initialization (both models, 25 epochs)
!python -u experiments/run_pretraining_ablation.py --pretraining imagenet --model all --epochs 25 --batch_size 4



In [ ]:
import pandas as pd

# Load the saved pretraining ablation summary
df = pd.read_csv('outputs/new_logs/02-Pretraining/02_pretraining_ablation_summary.csv')


imagenet_df = df[df['pretraining'] == 'imagenet']
print(imagenet_df[['model', 'pretraining', 'best_val_miou', 'best_val_dice', 'best_val_acc', 'best_val_biou']].to_string(index=False))

        model pretraining  best_val_miou  best_val_dice  best_val_acc  best_val_biou
deeplabv3plus    imagenet         0.6911         0.7773        0.9335         0.2613
 segformer_b2    imagenet         0.7745         0.8455        0.9574         0.4168


### 5.2 Run Pretraining Regime #2: Cityscapes Initialization (In-Domain Urban Scenes)

In [ ]:
!python -u experiments/run_pretraining_ablation.py --pretraining cityscapes --model all --epochs 25 --batch_size 4

In [2]:
import pandas as pd

# Load the saved pretraining ablation summary
df = pd.read_csv('outputs/new_logs/02-Pretraining/02_pretraining_ablation_summary.csv')

imagenet_df = df[df['pretraining'] == 'cityscapes']
print(imagenet_df[['model', 'pretraining', 'best_val_miou', 'best_val_dice', 'best_val_acc', 'best_val_biou']].to_string(index=False))

        model pretraining  best_val_miou  best_val_dice  best_val_acc  best_val_biou
deeplabv3plus  cityscapes         0.7770         0.8509        0.9559         0.3981
 segformer_b2  cityscapes         0.8047         0.8742        0.9607         0.4700


### 5.3 View Pretraining Domain Transfer Comparison Table

In [7]:
import os
import pandas as pd
summary_path = "./outputs/new_logs/02_pretraining_ablation_summary.csv"
if os.path.exists(summary_path):
    df_pretrain = pd.read_csv(summary_path)
    print("=== PRETRAINING ABLATION (IMAGENET VS. CITYSCAPES) ===")
    display(df_pretrain)
else:
    print("No pretraining ablation summary found yet. Run Section 5.1 and/or 5.2 above first.")

=== PRETRAINING ABLATION (IMAGENET VS. CITYSCAPES) ===


,model,pretraining,best_val_miou,best_val_dice,best_val_acc,best_val_biou,checkpoint
0,deeplabv3plus,imagenet,0.6911,0.7773,0.9335,0.2613,./outputs/new_checkpoints/02_pretrain_deeplabv...
1,segformer_b2,imagenet,0.7745,0.8455,0.9574,0.4168,./outputs/new_checkpoints/02_pretrain_segforme...
2,deeplabv3plus,cityscapes,0.7770,0.8509,0.9559,0.3981,./outputs/new_checkpoints/02_pretrain_deeplabv...
3,segformer_b2,cityscapes,0.8047,0.8742,0.9607,0.4700,./outputs/new_checkpoints/02_pretrain_segforme...


## 6. Loss Functions Study: 5 Individual Loss Ablations
Train both Cityscapes-pretrained models on each of the 5 distinct loss functions one by one:
1. **Cross-Entropy (CE)**: Distribution-based pixel classification baseline
2. **Lovász-Softmax**: Direct tractable convex surrogate IoU loss (Berman et al., CVPR 2018)
3. **Dice Loss**: Multi-class harmonic region overlap optimization
4. **Focal Loss**: Hard-negative pixel mining (Lin et al., ICCV 2017)
5. **JML (Jaccard Metric Loss)**: Soft-label metric surrogate (Wang et al., NeurIPS 2023)

> Each loss below is completely decoupled in its own cell. You can execute them in any order; results are safely appended and merged in `./outputs/new_logs/03_losses_ablation_summary.csv`.

### 6.1 Run Loss #1: Multi-class Cross-Entropy (CE)

In [ ]:
!python -u experiments/run_losses_ablation.py --loss ce --model all --epochs 25 --batch_size 4

In [18]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

cols = ['epoch', 'train_loss', 'train_miou', 'val_loss', 'val_miou', 'lr']
for m in ['deeplabv3plus', 'segformer_b2']:
    files = list(Path('./outputs').glob(f'**/*{m}*ce*history.csv'))
    if files:
        df = pd.read_csv(files[0])
        best = df.loc[[df['val_miou'].idxmax()], cols]
        display(Markdown(f"### **{m.upper()} Best Epoch (CE)**"))
        display(best)

### **DEEPLABV3PLUS Best Epoch (CE)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
21,22,0.1058,0.794,0.1274,0.777,0.0008


### **SEGFORMER_B2 Best Epoch (CE)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
15,16,0.0848,0.8335,0.1208,0.8047,0.000035


### 6.2 Run Loss #2: Lovász-Softmax Loss (Berman et al., CVPR 2018)

In [ ]:
!python -u experiments/run_losses_ablation.py --loss lovasz --model all --epochs 25 --batch_size 4

In [16]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

cols = ['epoch', 'train_loss', 'train_miou', 'val_loss', 'val_miou', 'lr']
for m in ['deeplabv3plus', 'segformer_b2']:
    files = list(Path('./outputs').glob(f'**/*{m}*lovasz*history.csv'))
    if files:
        df = pd.read_csv(files[0])
        best = df.loc[[df['val_miou'].idxmax()], cols]
        display(Markdown(f"### **{m.upper()} Best Epoch (Lovász-Softmax)**"))
        display(best)

### **DEEPLABV3PLUS Best Epoch (Lovász-Softmax)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
23,24,0.2427,0.8013,0.2485,0.7786,0.0004


### **SEGFORMER_B2 Best Epoch (Lovász-Softmax)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
22,23,0.1869,0.8425,0.2084,0.8146,0.000004


### 6.3 Run Loss #3: Multi-class Soft Dice Loss

In [ ]:
!python -u experiments/run_losses_ablation.py --loss dice --model all --epochs 25 --batch_size 4

In [19]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

cols = ['epoch', 'train_loss', 'train_miou', 'val_loss', 'val_miou', 'lr']
for m in ['deeplabv3plus', 'segformer_b2']:
    files = list(Path('./outputs').glob(f'**/*{m}*dice*history.csv'))
    if files:
        df = pd.read_csv(files[0])
        best = df.loc[[df['val_miou'].idxmax()], cols]
        display(Markdown(f"### **{m.upper()} Best Epoch (DICE)**"))
        display(best)

### **DEEPLABV3PLUS Best Epoch (DICE)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
22,23,0.1622,0.7946,0.1789,0.7779,0.0006


### **SEGFORMER_B2 Best Epoch (DICE)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
15,16,0.1266,0.8304,0.1406,0.8148,0.000035


### 6.4 Run Loss #4: Multi-class Focal Loss (Lin et al., ICCV 2017)

In [ ]:
!python -u experiments/run_losses_ablation.py --loss focal --model all --epochs 25 --batch_size 4

In [20]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

cols = ['epoch', 'train_loss', 'train_miou', 'val_loss', 'val_miou', 'lr']
for m in ['deeplabv3plus', 'segformer_b2']:
    files = list(Path('./outputs').glob(f'**/*{m}*focal*history.csv'))
    if files:
        df = pd.read_csv(files[0])
        best = df.loc[[df['val_miou'].idxmax()], cols]
        display(Markdown(f"### **{m.upper()} Best Epoch (Focal)**"))
        display(best)

### **DEEPLABV3PLUS Best Epoch (Focal)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
21,22,0.0446,0.7828,0.0572,0.7666,0.0008


### **SEGFORMER_B2 Best Epoch (Focal)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
9,10,0.0342,0.8173,0.0452,0.8047,0.000072


### 6.5 Run Loss #5: Jaccard Metric Loss (JML, Wang et al., NeurIPS 2023)

In [ ]:
!python -u experiments/run_losses_ablation.py --loss jml --model all --epochs 25 --batch_size 4

In [21]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

cols = ['epoch', 'train_loss', 'train_miou', 'val_loss', 'val_miou', 'lr']
for m in ['deeplabv3plus', 'segformer_b2']:
    files = list(Path('./outputs').glob(f'**/*{m}*jml*history.csv'))
    if files:
        df = pd.read_csv(files[0])
        best = df.loc[[df['val_miou'].idxmax()], cols]
        display(Markdown(f"### **{m.upper()} Best Epoch (JML)**"))
        display(best)

### **DEEPLABV3PLUS Best Epoch (JML)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
22,23,0.2517,0.7951,0.2562,0.779,0.0006


### **SEGFORMER_B2 Best Epoch (JML)**

,epoch,train_loss,train_miou,val_loss,val_miou,lr
15,16,0.2059,0.8345,0.2127,0.8153,0.000035


## 7. Goal-Driven Agent Search & Composite Loss Training
Formulate composite loss $\mathcal{L} = \lambda_{\text{CE}} \mathcal{L}_{\text{CE}} + \lambda_{\text{JML}} \mathcal{L}_{\text{JML}}$.
The workflow is split into two distinct, decoupled steps:
1. **Step 7.1 (Agent Search)**: Automated Bayesian Search Agent (Optuna TPE) explores and discovers optimal loss weights strictly on the validation set.
2. **Step 7.2 (Model Training)**: Trains DeepLabV3+ and SegFormer-B2 using the discovered optimal weights for full 25 epochs.

### 7.1 Goal-Driven Bayesian Optimization Agent (Independent Searches)
Runs Optuna TPE Bayesian search independently for each architecture with fast validation evaluations (AMP enabled, boundary-IoU bypassed during proxy search for ~35s/trial speed):


In [ ]:
# 7.1.1 Execute SegFormer-B2 Loss-Weight Search ONLY (~35s per trial)
!python -u experiments/search_segformer.py --agent_trials 15 --search_epochs 3 --data_root ./data/CamVid --log_dir ./outputs/new_logs

In [ ]:
# 7.1.2 Execute DeepLabV3+ Loss-Weight Search ONLY (~40s per trial)
!python -u experiments/search_deeplab.py --agent_trials 15 --search_epochs 3 --data_root ./data/CamVid --log_dir ./outputs/new_logs

In [25]:
import json, pandas as pd
from pathlib import Path

for m in ['segformer', 'deeplab']:
    w = next(Path('./outputs/new_logs').glob(f'**/*agent_weights_{m}*.json'), None)
    t = next(Path('./outputs/new_logs').glob(f'**/*agent_trials_{m}*.csv'), None)
    if w and t:
        cfg = json.load(open(w))
        print(f"\n=== {m.upper()} AGENT RESULTS ===")
        print(f"Optimal Weights: lambda_ce={cfg['lambda_ce']}, lambda_jml={cfg['lambda_jml']} (Best Trial #{cfg['best_trial']}, Val mIoU: {cfg['best_val_miou']:.4f})")
        display(pd.read_csv(t).sort_values('val_miou', ascending=False).head(5))


=== SEGFORMER AGENT RESULTS ===
Optimal Weights: lambda_ce=0.65, lambda_jml=1.1 (Best Trial #5, Val mIoU: 0.6679)


,trial,timestamp,lambda_ce,lambda_jml,val_miou,duration_seconds
4,5,2026-11-11 13:54:31,0.65,1.10,0.6679,96.92
5,6,2026-11-11 13:56:08,0.10,1.50,0.6592,88.40
0,1,2026-11-11 13:49:02,0.45,1.45,0.6553,93.98
3,4,2026-11-11 13:53:14,0.15,1.35,0.6180,77.03
12,13,2026-11-11 14:06:38,0.55,0.95,0.6130,75.19



=== DEEPLAB AGENT RESULTS ===
Optimal Weights: lambda_ce=0.75, lambda_jml=0.9500000000000001 (Best Trial #2, Val mIoU: 0.6203)


,trial,timestamp,lambda_ce,lambda_jml,val_miou,duration_seconds
1,2,2026-11-11 14:11:57,0.75,0.95,0.6203,53.12
4,5,2026-11-11 14:14:36,0.65,1.10,0.6101,53.42
3,4,2026-11-11 14:13:43,0.15,1.35,0.6063,53.17
5,6,2026-11-11 14:15:30,0.10,1.50,0.5902,53.63
0,1,2026-11-11 14:11:02,0.45,1.45,0.5892,54.66


### 7.2 Full Composite Loss Model Training (25 Epochs) on semantic transfer from cityscapes on seeds(42,123,456)
Trains DeepLabV3+ and SegFormer-B2 with their respective agent-discovered optimal weights (auto-loaded from `04_agent_weights_deeplab.json` and `04_agent_weights_segformer.json`). Full evaluation metrics (mIoU, Dice/F1, Pixel Acc, Boundary IoU) are computed and checkpoints are saved under `outputs/new_checkpoints/`.

In [ ]:
!python -u experiments/train_semantic_transfer.py --model segformer_b2 --full_epochs 25

In [ ]:
!python -u experiments/train_semantic_transfer.py --model deeplabv3plus --full_epochs 25 --batch_size 4

In [27]:
import pandas as pd
from pathlib import Path

csv_p = next(Path('./outputs').glob('**/*semantic_transfer_summary*.csv'), None) or next(Path('./outputs').glob('**/*Semantic_Transfer*.csv'))

print("=== SEMANTIC EMBEDDING TRANSFER TRAINING COMPLETE ===")
display(pd.read_csv(csv_p))

=== SEMANTIC EMBEDDING TRANSFER TRAINING COMPLETE ===


,model,lambda_ce,lambda_jml,best_val_miou,best_val_dice,best_val_acc,best_val_biou,checkpoint
0,segformer_b2,0.65,1.10,0.8178,0.8837,0.9651,0.4825,./outputs/checkpoints/04_semantic_segformer_b2...
1,deeplabv3plus,0.75,0.95,0.7796,0.8523,0.9560,0.3965,./outputs/checkpoints/04_semantic_deeplabv3plu...


In [28]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*03_losses_ablation_summary.csv')))
cols = ['model', 'loss', 'best_val_miou', 'best_val_dice', 'best_val_acc', 'best_val_biou']
display(df.sort_values(['model', 'best_val_miou'], ascending=[True, False])[cols])

,model,loss,best_val_miou,best_val_dice,best_val_acc,best_val_biou
10,deeplabv3plus,composite,0.7796,0.8523,0.9560,0.3965
8,deeplabv3plus,jml,0.7790,0.8514,0.9556,0.3934
2,deeplabv3plus,lovasz,0.7786,0.8510,0.9551,0.3998
4,deeplabv3plus,dice,0.7779,0.8522,0.9552,0.3887
0,deeplabv3plus,ce,0.7770,0.8509,0.9559,0.3981
6,deeplabv3plus,focal,0.7666,0.8436,0.9521,0.3810
11,segformer_b2,composite,0.8178,0.8837,0.9651,0.4825
9,segformer_b2,jml,0.8153,0.8824,0.9638,0.4704
5,segformer_b2,dice,0.8148,0.8832,0.9639,0.4705
3,segformer_b2,lovasz,0.8146,0.8808,0.9632,0.4746


In [ ]:
# Seed 2 (123)
!python -u experiments/train_semantic_transfer.py --model all --seed 123 --full_epochs 25 --batch_size 4



In [32]:
import json, pandas as pd
from pathlib import Path

rows = [json.load(open(p)) for p in Path('./outputs').glob('**/*seed123*summary.json')]
cols = ['model', 'seed', 'lambda_ce', 'lambda_jml', 'best_val_miou', 'best_val_dice', 'best_val_acc', 'best_val_biou']

print("=== SEED 123 Composite Training RESULTS ===")
display(pd.DataFrame(rows)[cols])

=== SEED 123 Composite Training RESULTS ===


,model,seed,lambda_ce,lambda_jml,best_val_miou,best_val_dice,best_val_acc,best_val_biou
0,deeplabv3plus,123,0.75,0.95,0.7817,0.8544,0.9562,0.4020
1,segformer_b2,123,0.65,1.10,0.8178,0.8849,0.9639,0.4815


In [ ]:
# Seed 3 (456)
!python -u experiments/train_semantic_transfer.py --model all --seed 456 --full_epochs 25 --batch_size 4

In [31]:
import json, pandas as pd
from pathlib import Path

rows = [json.load(open(p)) for p in Path('./outputs').glob('**/*seed456*summary.json')]
cols = ['model', 'seed', 'lambda_ce', 'lambda_jml', 'best_val_miou', 'best_val_dice', 'best_val_acc', 'best_val_biou']

print("=== SEED 456 Composite Training RESULTS ===")
display(pd.DataFrame(rows)[cols])

=== SEED 456 Composite Training RESULTS ===


,model,seed,lambda_ce,lambda_jml,best_val_miou,best_val_dice,best_val_acc,best_val_biou
0,deeplabv3plus,456,0.75,0.95,0.7823,0.8545,0.9565,0.4023
1,segformer_b2,456,0.65,1.10,0.8152,0.8815,0.9641,0.4804


### 7.3 Losses Ablation 

In [ ]:
!python -u experiments/run_losses_ablation.py --eval_only --split val

In [ ]:
!python -u experiments/run_losses_ablation.py --eval_only --split test

In [58]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*03_losses_ablation_summary.csv')))

# Format percentages & boundary IoU
for col in ['best_val_miou', 'best_val_acc', 'best_val_dice']:
    df[col] = (df[col] * 100).map('{:.2f}%'.format)
df['best_val_biou'] = df['best_val_biou'].map('{:.4f}'.format)

df.rename(columns={
    'best_val_miou': 'Val mIoU',
    'best_val_acc': 'Pixel Acc',
    'best_val_dice': 'Dice/F1',
    'best_val_biou': 'Boundary IoU'
}, inplace=True)

display(df[['model', 'loss', 'Val mIoU', 'Pixel Acc', 'Dice/F1', 'Boundary IoU']])

,model,loss,Val mIoU,Pixel Acc,Dice/F1,Boundary IoU
0,deeplabv3plus,ce,77.70%,95.59%,85.09%,0.3981
1,segformer_b2,ce,80.47%,96.07%,87.42%,0.4700
2,deeplabv3plus,lovasz,77.86%,95.51%,85.10%,0.3998
3,segformer_b2,lovasz,81.46%,96.32%,88.08%,0.4746
4,deeplabv3plus,dice,77.79%,95.52%,85.22%,0.3887
5,segformer_b2,dice,81.48%,96.39%,88.32%,0.4705
6,deeplabv3plus,focal,76.66%,95.21%,84.36%,0.3810
7,segformer_b2,focal,80.47%,96.11%,87.61%,0.4684
8,deeplabv3plus,jml,77.90%,95.56%,85.14%,0.3934
9,segformer_b2,jml,81.53%,96.38%,88.24%,0.4704


In [ ]:
!python -u experiments/run_losses_ablation.py --eval_only --split test

In [60]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*Table_09_Losses_Ablation_Test*.csv')))

for col in ['test_mIoU', 'pixel_accuracy', 'dice_f1', 'precision', 'recall']:
    df[col] = (df[col] * 100).map('{:.2f}%'.format)
df['boundary_iou'] = df['boundary_iou'].map('{:.4f}'.format)

display(df[['model', 'loss', 'test_mIoU', 'pixel_accuracy', 'dice_f1', 'precision', 'recall', 'boundary_iou']])

,model,loss,test_mIoU,pixel_accuracy,dice_f1,precision,recall,boundary_iou
0,deeplabv3plus,ce,71.99%,93.12%,82.01%,83.45%,81.05%,0.3757
1,deeplabv3plus,lovasz,72.07%,92.58%,82.25%,82.36%,82.26%,0.3664
2,deeplabv3plus,dice,71.46%,92.43%,81.81%,82.98%,80.94%,0.3528
3,deeplabv3plus,focal,71.41%,92.75%,81.60%,82.41%,81.10%,0.3671
4,deeplabv3plus,jml,72.07%,92.63%,82.22%,83.34%,81.34%,0.3588
5,deeplabv3plus,composite,72.30%,92.96%,82.35%,83.59%,81.44%,0.3727
6,segformer_b2,ce,74.51%,93.70%,83.97%,84.84%,83.44%,0.4239
7,segformer_b2,lovasz,74.33%,93.34%,83.92%,84.78%,83.21%,0.4261
8,segformer_b2,dice,73.84%,93.14%,83.55%,84.18%,83.03%,0.4188
9,segformer_b2,focal,73.98%,93.35%,83.67%,84.38%,83.16%,0.4189


## 8. Improvements Ablation: 6 Stages Evaluated One by One (3 Seeds)
Execute and evaluate each improvement individually or cumulatively across 3 random seeds (`42, 123, 456`):
1. **Stage 1**: Baseline Composite Loss (CE + JML) Base Model (25 epochs)
2. **Stage 2**: Native 720×960 Sliding Window Inference with 2D Gaussian Blending
3. **Stage 3**: Multi-Scale TTA Pyramid Evaluation (scales [0.75, 1.0, 1.25] + horizontal flip)
4. **Stage 4**: Class Prior Calibration & Temperature Scaling (log-prior shift)



> Each stage can be run one by one in its own cell below. Results are automatically merged into `./outputs/new_logs/05_improvements_ablation_test_steps.csv`.

### 8.1 Improvement Stage 1: Baseline Composite Loss (CE + JML)

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 1 --model all --seed all --split val --epochs 25 --batch_size 4 

In [39]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*Table_13*.csv')))
s1 = df[df['stage'].str.contains('1. Baseline')].copy()
s1['split'] = 'Test'

out = s1.groupby(['model', 'split', 'type', 'stage']).agg(
    **{'mIoU (Mean ± Std)': ('mIoU', lambda x: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%"),
       'Boundary IoU (Mean ± Std)': ('boundary_iou', lambda x: f"{x.mean():.4f} ± {x.std():.4f}")}
).reset_index()
display(out)

,model,split,type,stage,mIoU (Mean ± Std),Boundary IoU (Mean ± Std)
0,deeplabv3plus,Test,Cumulative,1. Baseline Composite Loss,72.18 ± 0.11%,0.3731 ± 0.0005
1,segformer_b2,Test,Cumulative,1. Baseline Composite Loss,74.59 ± 0.29%,0.4301 ± 0.0012


### 8.2 Improvement Stage 2: Native 720×960 Sliding Window Inference

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 2 --model all --seed all --split val

In [42]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*improvements_ablation_val_steps.csv')))
s2 = df[df['stage'].str.contains('Sliding Window')].copy()
s2['split'] = 'Validation'

out = s2.groupby(['model', 'split', 'type', 'stage']).agg(
    **{'mIoU (Mean ± Std)': ('mIoU', lambda x: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%" if len(x)>1 else f"{x.mean()*100:.2f}%"),
       'Boundary IoU (Mean ± Std)': ('boundary_iou', lambda x: f"{x.mean():.4f} ± {x.std():.4f}" if len(x)>1 else f"{x.mean():.4f}")}
).reset_index()

display(out)

,model,split,type,stage,mIoU (Mean ± Std),Boundary IoU (Mean ± Std)
0,deeplabv3plus,Validation,Cumulative,2. + Native Sliding Window,82.51%,0.3839
1,segformer_b2,Validation,Cumulative,2. + Native Sliding Window,84.97%,0.4270


### 8.3 Improvement Stage 3: Multi-Scale TTA Pyramid Evaluation

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 3 --model all --seed all --split val

In [43]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*improvements_ablation_val_steps.csv')))
s3 = df[df['stage'].str.contains('Multi-Scale TTA')].copy()
s3['split'] = 'Validation'

out = s3.groupby(['model', 'split', 'type', 'stage']).agg(
    **{'mIoU (Mean ± Std)': ('mIoU', lambda x: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%" if len(x)>1 else f"{x.mean()*100:.2f}%"),
       'Boundary IoU (Mean ± Std)': ('boundary_iou', lambda x: f"{x.mean():.4f} ± {x.std():.4f}" if len(x)>1 else f"{x.mean():.4f}")}
).reset_index()

display(out)

,model,split,type,stage,mIoU (Mean ± Std),Boundary IoU (Mean ± Std)
0,deeplabv3plus,Validation,Cumulative,3. + Multi-Scale TTA,83.08%,0.3923
1,segformer_b2,Validation,Cumulative,3. + Multi-Scale TTA,85.46%,0.4359


### 8.4 Improvement Stage 4: Class Prior Calibration & Temperature Scaling

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 4 --model all --seed all --split val

In [47]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*improvements_ablation_val_steps.csv')))
s4 = df[df['stage'].str.contains('Class Prior Calibration')].copy()
s4['split'] = 'Validation'

out = s4.groupby(['model', 'split', 'type', 'stage']).agg(
    **{'mIoU (Mean ± Std)': ('mIoU', lambda x: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%" if len(x)>1 else f"{x.mean()*100:.2f}%"),
       'Boundary IoU (Mean ± Std)': ('boundary_iou', lambda x: f"{x.mean():.4f} ± {x.std():.4f}" if len(x)>1 else f"{x.mean():.4f}")}
).reset_index()

display(out)

,model,split,type,stage,mIoU (Mean ± Std),Boundary IoU (Mean ± Std)
0,deeplabv3plus,Validation,Cumulative,4. + Class Prior Calibration (Full Peak Pipeline),83.21%,0.3927
1,segformer_b2,Validation,Cumulative,4. + Class Prior Calibration (Full Peak Pipeline),85.46%,0.4345


### 8.5 Print Consolidated Improvements Ablation Table (Mean ± Std Across Seeds)

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage summary --split val

In [51]:
import pandas as pd
from pathlib import Path

csv_p = next(Path('./outputs').glob('**/*improvements_ablation_val_steps.csv'), None) or next(Path('./outputs').glob('**/*Table_12*.csv'))
df = pd.read_csv(csv_p)

# Filter for the cumulative improvements ladder
ladder = df[df['type'] == 'Cumulative'].copy()
ladder['split'] = 'Validation'

cols = ['model', 'split', 'stage', 'mIoU', 'pixel_accuracy', 'dice_f1', 'boundary_iou']
print("=== IMPROVEMENTS ABLATION SUMMARY (VALIDATION SPLIT) ===")
display(ladder[cols].reset_index(drop=True))

=== IMPROVEMENTS ABLATION SUMMARY (VALIDATION SPLIT) ===


,model,split,stage,mIoU,pixel_accuracy,dice_f1,boundary_iou
0,segformer_b2,Validation,1. Baseline Composite Loss,0.8178,0.9651,0.8837,0.4825
1,deeplabv3plus,Validation,1. Baseline Composite Loss,0.7796,0.9560,0.8523,0.3965
2,segformer_b2,Validation,2. + Native Sliding Window,0.8497,0.9692,0.9109,0.4270
3,deeplabv3plus,Validation,2. + Native Sliding Window,0.8251,0.9649,0.8890,0.3839
4,segformer_b2,Validation,3. + Multi-Scale TTA,0.8546,0.9704,0.9139,0.4359
5,deeplabv3plus,Validation,3. + Multi-Scale TTA,0.8308,0.9663,0.8920,0.3923
6,segformer_b2,Validation,4. + Class Prior Calibration (Full Peak Pipeline),0.8546,0.9702,0.9144,0.4345
7,deeplabv3plus,Validation,4. + Class Prior Calibration (Full Peak Pipeline),0.8321,0.9664,0.8934,0.3927


## 9. Improvements Ablation: 6 Stages Evaluated One by One (3 Seeds)(Test Split)

### 9.1 Improvement Stage 1: Baseline Composite Loss (CE + JML)


In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 1 --model all --seed all --epochs 25 --batch_size 4

In [38]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*improvements_ablation_val_steps.csv')))
s1 = df[df['stage'].str.contains('1. Baseline')].copy()
s1['split'] = 'Validation'

cols = ['model', 'split', 'seed', 'type', 'stage', 'mIoU', 'pixel_accuracy', 'dice_f1', 'boundary_iou']
display(s1[cols].reset_index(drop=True))

,model,split,seed,type,stage,mIoU,pixel_accuracy,dice_f1,boundary_iou
0,segformer_b2,Validation,42,Cumulative,1. Baseline Composite Loss,0.8178,0.9651,0.8837,0.4825
1,deeplabv3plus,Validation,42,Cumulative,1. Baseline Composite Loss,0.7796,0.9560,0.8523,0.3965


### 9.2 Improvement Stage 2: Native 720×960 Sliding Window Inference

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 2 --model all --seed all

In [41]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*Table_13*.csv')))
s2 = df[df['stage'].str.contains('Sliding Window')].copy()
s2['split'] = 'Test'

out = s2.groupby(['model', 'split', 'type', 'stage']).agg(
    **{'mIoU (Mean ± Std)': ('mIoU', lambda x: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%"),
       'Boundary IoU (Mean ± Std)': ('boundary_iou', lambda x: f"{x.mean():.4f} ± {x.std():.4f}")}
).reset_index()

display(out)

,model,split,type,stage,mIoU (Mean ± Std),Boundary IoU (Mean ± Std)
0,deeplabv3plus,Test,Cumulative,2. + Native Sliding Window,76.31 ± 0.06%,0.3486 ± 0.0012
1,segformer_b2,Test,Cumulative,2. + Native Sliding Window,78.08 ± 0.47%,0.3918 ± 0.0016


### 9.3 Improvement Stage 3: Multi-Scale TTA Pyramid Evaluation

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 3 --model all --seed all

In [44]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*Table_13*.csv')))
s3 = df[df['stage'].str.contains('Multi-Scale TTA')].copy()
s3['split'] = 'Test'

out = s3.groupby(['model', 'split', 'type', 'stage']).agg(
    **{'mIoU (Mean ± Std)': ('mIoU', lambda x: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%"),
       'Boundary IoU (Mean ± Std)': ('boundary_iou', lambda x: f"{x.mean():.4f} ± {x.std():.4f}")}
).reset_index()

display(out)

,model,split,type,stage,mIoU (Mean ± Std),Boundary IoU (Mean ± Std)
0,deeplabv3plus,Test,Cumulative,3. + Multi-Scale TTA,76.95 ± 0.04%,0.3546 ± 0.0015
1,segformer_b2,Test,Cumulative,3. + Multi-Scale TTA,78.77 ± 0.34%,0.3997 ± 0.0014


### 9.4 Improvement Stage 4: Class Prior Calibration & Temperature Scaling

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage 4 --model all --seed all

In [46]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*Table_13*.csv')))
s4 = df[df['stage'].str.contains('Class Prior Calibration')].copy()
s4['split'] = 'Test'

out = s4.groupby(['model', 'split', 'type', 'stage']).agg(
    **{'mIoU (Mean ± Std)': ('mIoU', lambda x: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%"),
       'Boundary IoU (Mean ± Std)': ('boundary_iou', lambda x: f"{x.mean():.4f} ± {x.std():.4f}")}
).reset_index()

display(out)

,model,split,type,stage,mIoU (Mean ± Std),Boundary IoU (Mean ± Std)
0,deeplabv3plus,Test,Cumulative,4. + Class Prior Calibration (Full Peak Pipeline),77.00 ± 0.05%,0.3552 ± 0.0015
1,segformer_b2,Test,Cumulative,4. + Class Prior Calibration (Full Peak Pipeline),78.76 ± 0.34%,0.3987 ± 0.0014


### 9.5 Print Consolidated Improvements Ablation Table (Mean ± Std Across Seeds)

In [ ]:
!python -u experiments/run_improvements_ablation.py --stage summary

In [49]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(next(Path('./outputs').glob('**/*Table_05*.csv')))
print("=== IMPROVEMENTS ABLATION PROGRESS (MEAN ± STD) ===")
display(df)

=== IMPROVEMENTS ABLATION PROGRESS (MEAN ± STD) ===


,Model Architecture,Progressive Stage,Test mIoU (%),Pixel Acc (%),Mean Dice (%),Boundary IoU
0,DeepLabV3+ (ResNet-101),1. Baseline Composite Loss,72.18 ± 0.11%,92.87 ± 0.08%,82.28 ± 0.07%,0.3731 ± 0.0005
1,DeepLabV3+ (ResNet-101),2. + Native Sliding Window,76.31 ± 0.06%,93.81 ± 0.06%,85.53 ± 0.02%,0.3486 ± 0.0012
2,DeepLabV3+ (ResNet-101),3. + Multi-Scale TTA Pyramid,76.95 ± 0.04%,94.02 ± 0.05%,86.00 ± 0.03%,0.3546 ± 0.0015
3,DeepLabV3+ (ResNet-101),4. + Class Prior Shift (Full Pipeline),77.00 ± 0.05%,94.00 ± 0.06%,86.06 ± 0.02%,0.3552 ± 0.0015
4,SegFormer-B2 (MiT-B2),1. Baseline Composite Loss,74.59 ± 0.29%,93.55 ± 0.15%,84.09 ± 0.17%,0.4301 ± 0.0012
5,SegFormer-B2 (MiT-B2),2. + Native Sliding Window,78.08 ± 0.47%,94.31 ± 0.14%,86.80 ± 0.33%,0.3918 ± 0.0016
6,SegFormer-B2 (MiT-B2),3. + Multi-Scale TTA Pyramid,78.77 ± 0.34%,94.53 ± 0.13%,87.28 ± 0.23%,0.3997 ± 0.0014
7,SegFormer-B2 (MiT-B2),4. + Class Prior Shift (Full Pipeline),78.76 ± 0.34%,94.49 ± 0.13%,87.28 ± 0.23%,0.3987 ± 0.0014


## 10. Final Comprehensive Test Set Evaluation (3 Seeds Mean ± Std)
Evaluate final best models on the official test set (233 images) with all required metrics, per-class IoU tables, and normalized confusion matrix heatmaps.

In [ ]:
!python -u experiments/run_final_test.py --data_root ./data/CamVid --log_dir ./outputs/new_logs

In [55]:
import pandas as pd

# 1. Per-seed & Aggregated Overall Test Metrics
df = pd.read_csv("outputs/new_logs/Improvements/Test Set/05_improvements_ablation_test_steps.csv")
s4 = df[df["stage"].str.contains("Class Prior", na=False)]

print("=== 1. PER-SEED TEST METRICS (SEEDS 42, 123, 456) ===")
cols = ["model", "seed", "mIoU", "pixel_accuracy", "dice_f1", "precision", "recall", "boundary_iou"]
display(s4[cols].assign(**{c: (s4[c]*100).round(2).astype(str)+'%' for c in cols[2:-1]}, boundary_iou=s4['boundary_iou'].round(4)))

print("\n=== 2. MEAN ± STD ACROSS 3 SEEDS ===")
agg = s4.groupby("model").agg(
    **{k: (col, lambda x, c=col: f"{x.mean()*100:.2f} ± {x.std()*100:.2f}%" if c != 'boundary_iou' else f"{x.mean():.4f} ± {x.std():.4f}")
       for k, col in [('mIoU (Mean ± Std)', 'mIoU'), ('Pixel Acc', 'pixel_accuracy'), ('Dice / F1', 'dice_f1'), ('Precision', 'precision'), ('Recall', 'recall'), ('Boundary IoU', 'boundary_iou')]}
).reset_index()
display(agg)

# 2. Complete Per-Class IoU Breakdown
print("\n=== 3. PER-CLASS TEST IoU BREAKDOWN (ALL 11 CLASSES) ===")
display(pd.read_csv("outputs/Figures_and_Tables/Tables/Table_07_Per_Class_IoU_Breakdown.csv"))

=== 1. PER-SEED TEST METRICS (SEEDS 42, 123, 456) ===


,model,seed,mIoU,pixel_accuracy,dice_f1,precision,recall,boundary_iou
18,segformer_b2,42,78.59%,94.36%,87.2%,86.84%,87.62%,0.3977
19,segformer_b2,123,78.55%,94.49%,87.1%,87.41%,86.93%,0.3982
20,segformer_b2,456,79.15%,94.62%,87.54%,88.05%,87.21%,0.4003
21,deeplabv3plus,42,77.05%,94.05%,86.07%,87.37%,85.16%,0.3547
22,deeplabv3plus,123,77.01%,93.94%,86.07%,86.97%,85.47%,0.3540
23,deeplabv3plus,456,76.95%,94.02%,86.03%,87.1%,85.26%,0.3568



=== 2. MEAN ± STD ACROSS 3 SEEDS ===


,model,mIoU (Mean ± Std),Pixel Acc,Dice / F1,Precision,Recall,Boundary IoU
0,deeplabv3plus,77.00 ± 0.05%,94.00 ± 0.06%,86.06 ± 0.02%,87.15 ± 0.20%,85.30 ± 0.16%,0.3552 ± 0.0015
1,segformer_b2,78.76 ± 0.34%,94.49 ± 0.13%,87.28 ± 0.23%,87.43 ± 0.61%,87.25 ± 0.35%,0.3987 ± 0.0014



=== 3. PER-CLASS TEST IoU BREAKDOWN (ALL 11 CLASSES) ===


,Class Index,Semantic Class,Train Frequency (%),DeepLabV3+ Test IoU (%),SegFormer-B2 Test IoU (%),Absolute Delta
0,0,Sky,17.54%,92.41%,93.12%,+0.71%
1,1,Building,24.22%,86.75%,88.45%,+1.70%
2,2,Pole,1.02%,41.20%,44.80%,+3.60%
3,3,Road,32.96%,95.80%,96.42%,+0.62%
4,4,Pavement / Sidewalk,4.67%,84.15%,86.10%,+1.95%
5,5,Tree / Vegetation,10.12%,82.30%,84.75%,+2.45%
6,6,SignSymbol,1.22%,51.40%,54.90%,+3.50%
7,7,Fence,1.17%,46.20%,49.80%,+3.60%
8,8,Car,6.11%,88.65%,90.25%,+1.60%
9,9,Pedestrian,0.67%,66.45%,69.85%,+3.40%
